# CarMax non-prime ABS: Tier 1 composition, dataset inventory, tier visibility

Three questions:
- **A.** What is in the non-prime trust filings (*CarMax Select Receivables Trusts*)? Does the data support the view that they are largely **Tier 1** loans?
- **B.** Which CarMax datasets have we ingested, and can each loan be tied to its trust?
- **C.** Is there a loan-level tier rating? If not, can tier be modelled from observable characteristics?

**Sources**
- Loan-level data in Snowflake: `BOTS.AUTOABS`, `BOTS.KMX`.
- CarMax 10-K FY2026 and 10-Q Q2 FY2027, via the SRS documents tool ([dashboard](https://apps.srsinvest.com/srs-documents-tool/0ea2d1ff795a4b45/)).
- Select trust prospectuses (424B5) on SEC EDGAR.

### Summary (from the 2026-10-06 run)

**Part A — the Tier 1 view is supported, with a caveat.**

1. **CarMax's tiers are defined by who lends, not by credit score.**
   - Tier 1 is CarMax Auto Finance's (CAF) *core* lending.
   - Tier 2 and Tier 3 are third-party lenders. CAF also now lends into the Tier 2/3 credit band itself.
   - CAF's scorecard grades loans A, B, C, and FICO is "not a significant factor" in it.
2. **The prospectuses define the Select pools as two parts:**
   - CAF **core-portfolio** loans with FICO < 650 (Tier 1);
   - plus loans **originated outside the core portfolio** (CAF's Tier 2/3 programme).
   - The prospectuses do not quantify the split. I only had extracted text to check, so confirm against the full documents.
3. **Our loan-level data reconciles with the prospectuses.** Loan counts and weighted-average (WA) FICO match exactly for 2025-A and 2025-B; 2024-A is 2 loans over.
4. **84–92% of each Select pool's balance fits the core-portfolio rule** (FICO < 650). The rest is:
   - loans with no FICO at all (3–6%), carrying a higher APR (~18.5–19.4% vs ~16%);
   - FICO ≥ 650 loans.
5. **10-K cross-check.** The 10-K splits the allowance released on the Sept-2025 non-prime sale (= Select 2025-B, about $930M) as **$30.3M Tier 1 / $11.9M Tier 2/3**. At the Tier 2/3 allowance rate (about 13.7%), that implies **Tier 2/3 ≈ 9% of the pool, Tier 1 ≈ 91%**.
6. **Caveat.** Tier cannot be assigned loan by loan. And CAF plans to raise its Tier 2 share from FY27, so future Select pools may hold more Tier 2.

**Part B — inventory and trust linkage.**
- Every ABS row carries `TRUST_NAME`.
- About 74k asset numbers recur across two trusts, so the loan key must be **(trust_name, assetnumber)**.
- The raw tables contain about 2.6% duplicate rows on (trust, asset, period).
- The retail and alt-data sets are aggregates with no loan or trust key.

**Part C — no tier field anywhere.**
- The SEC loan-level format (EX-102) has no tier or grade field, and neither do any of our tables.
- A proxy model can separate Select-shelf loans from prime-shelf loans. However, the shelf is not the tier, and the internal scorecard is not observable.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import snowflake.connector

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

conn = snowflake.connector.connect(connection_name=os.getenv("SNOWFLAKE_CONNECTION", "default"))

def q(sql: str) -> pd.DataFrame:
    df = conn.cursor().execute(sql).fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df

def try_q(sql: str) -> pd.DataFrame:
    try:
        return q(sql)
    except Exception as e:  # role may lack access to some databases
        return pd.DataFrame({"error": [str(e).splitlines()[0]]})

RAW = "BOTS.AUTOABS.CARMAX_DETAIL"         # full history per trust (from trust inception)
SEC2 = "BOTS.AUTOABS.CARMAX_SEC2_DETAIL"   # second scrape, periods 2025-08 onwards
KMXD = "BOTS.KMX.DETAIL"
PROC = "BOTS.KMX.ABS_COMBINED_PROCESSED"
FED = "BOTS.KMX.FED_FUND_RATES"
SELECT_LIKE = "trust_name ILIKE '%Select%'"

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...


## Part A — Non-prime (Select) trust contents and the Tier 1 test

### A1. Scope: which tables hold the Select trusts
`RAW` covers each trust from its first month (cutoff), so A uses `RAW`. Origination attributes come from each loan's **first** observation, with duplicates removed via `QUALIFY`.

In [2]:
q(f'''
SELECT src, trust_name, COUNT(*) AS n_rows, MIN(reportingperiodendingdate) AS first_period, MAX(reportingperiodendingdate) AS last_period
FROM (
    SELECT 'RAW' AS src, trust_name, reportingperiodendingdate FROM {RAW} WHERE {SELECT_LIKE}
    UNION ALL SELECT 'SEC2', trust_name, reportingperiodendingdate FROM {SEC2} WHERE {SELECT_LIKE}
    UNION ALL SELECT 'KMX.DETAIL', trust_name, reportingperiodendingdate FROM {KMXD} WHERE {SELECT_LIKE}
)
GROUP BY 1, 2 ORDER BY 2, 1
''')

,src,trust_name,n_rows,first_period,last_period
0,KMX.DETAIL,CarMax Select Receivables Trust 2024-A,164076,2025-08-31,2025-12-31
1,RAW,CarMax Select Receivables Trust 2024-A,656317,2024-05-31,2025-12-31
2,SEC2,CarMax Select Receivables Trust 2024-A,196891,2025-08-31,2026-01-31
3,KMX.DETAIL,CarMax Select Receivables Trust 2025-A,211495,2025-08-31,2025-12-31
4,RAW,CarMax Select Receivables Trust 2025-A,498326,2025-02-28,2025-12-31
5,SEC2,CarMax Select Receivables Trust 2025-A,253794,2025-08-31,2026-01-31
6,KMX.DETAIL,CarMax Select Receivables Trust 2025-B,195200,2025-09-30,2025-12-31
7,RAW,CarMax Select Receivables Trust 2025-B,277705,2025-08-31,2025-12-31
8,SEC2,CarMax Select Receivables Trust 2025-B,244000,2025-09-30,2026-01-31


In [8]:
# One row per loan at its first observation (cutoff month), all CarMax trusts
first_obs = q(f'''
WITH f AS (
    SELECT * FROM {RAW}
    QUALIFY ROW_NUMBER() OVER (PARTITION BY trust_name, assetnumber ORDER BY reportingperiodendingdate, insertupdatetime DESC) = 1
),
ff AS (SELECT DATE_TRUNC('month', observation_date) AS m, AVG(dff) AS dff FROM {FED} GROUP BY 1)
SELECT f.trust_name,
       IFF(f.trust_name ILIKE '%Select%', 'Select', 'Prime') AS shelf,
       f.assetnumber,
       f.reportingperiodendingdate                            AS cutoff_period,
       f.originationdate,
       f.reportingperiodbeginningloanbalanceamount            AS beg_bal,
       f.reportingperiodactualendbalanceamount                AS end_bal,
       f.originalloanamount                                   AS orig_amt,
       f.originalloanterm                                     AS term,
       f.originalinterestratepercentage * 100                 AS apr,          -- stored as a fraction
       NULLIF(f.obligorcreditscore, 0)                        AS fico,
       f.originalloanamount / NULLIF(f.vehiclevalueamount, 0) AS ltv,
       f.paymenttoincomepercentage * 100                      AS pti,          -- stored as a fraction
       IFF(f.coobligorindicator, 1, 0)                        AS coobligor,
       f.obligorincomeverificationlevelcode                   AS income_verif,
       f.vehicletypecode                                      AS vehicle_type,
       YEAR(f.originationdate) - f.vehiclemodelyear           AS vehicle_age,
       f.obligorgeographiclocation                            AS state,
       f.originalinterestratepercentage * 100 - ff.dff        AS apr_spread
FROM f LEFT JOIN ff ON ff.m = DATE_TRUNC('month', f.originationdate)
''')
sel = first_obs[first_obs.shelf == "Select"]
first_obs.groupby("trust_name").size().rename("loans").to_frame().T

trust_name,CarMax Auto Owner Trust 2021-4,CarMax Auto Owner Trust 2022-1,CarMax Auto Owner Trust 2022-2,CarMax Auto Owner Trust 2022-3,CarMax Auto Owner Trust 2022-4,CarMax Auto Owner Trust 2023-1,CarMax Auto Owner Trust 2023-2,CarMax Auto Owner Trust 2023-3,CarMax Auto Owner Trust 2023-4,CarMax Auto Owner Trust 2024-1,CarMax Auto Owner Trust 2024-2,CarMax Auto Owner Trust 2024-3,CarMax Auto Owner Trust 2024-4,CarMax Auto Owner Trust 2025-1,CarMax Auto Owner Trust 2025-2,CarMax Auto Owner Trust 2025-3,CarMax Auto Owner Trust 2025-4,CarMax Select Receivables Trust 2024-A,CarMax Select Receivables Trust 2025-A,CarMax Select Receivables Trust 2025-B
loans,66752,86274,74036,70744,67667,75490,79331,76518,77131,81506,85552,71949,146505,82495,79102,86717,63112,32818,42299,48800


### A2. Field inventory: Select vs prime
Which fields are populated, and which code fields actually vary.

In [9]:
code_cols = ["income_verif", "vehicle_type", "coobligor"]
for c in code_cols:
    display(pd.crosstab(first_obs[c].fillna(-1), first_obs.shelf, normalize="columns").style.format("{:.1%}").set_caption(c))

fill = first_obs.groupby("shelf").agg(lambda s: s.notna().mean()).T
fill.style.format("{:.1%}").set_caption("Fill rate by shelf (first observation)")

shelf,Prime,Select
income_verif,,
2,99.5%,91.4%
3,0.5%,8.6%


shelf,Prime,Select
vehicle_type,,
1,36.4%,38.8%
2,9.1%,6.3%
3,44.0%,41.9%
98,2.8%,1.7%
99,7.8%,11.3%


shelf,Prime,Select
coobligor,,
0,70.5%,72.1%
1,29.5%,27.9%


shelf,Prime,Select
trust_name,100.0%,100.0%
assetnumber,100.0%,100.0%
cutoff_period,100.0%,100.0%
originationdate,100.0%,100.0%
beg_bal,100.0%,100.0%
end_bal,100.0%,100.0%
orig_amt,100.0%,100.0%
term,100.0%,100.0%
apr,100.0%,100.0%
fico,99.2%,92.9%


In [10]:
# Fields that are constant across the whole CarMax tape (no discriminating power)
q(f'''
SELECT IFF({SELECT_LIKE}, 'Select', 'Prime') AS shelf,
       COUNT(DISTINCT originatorname) AS originatorname, ANY_VALUE(originatorname) AS originator_value,
       COUNT(DISTINCT subvented) AS subvented, COUNT(DISTINCT underwritingindicator) AS underwritingindicator,
       COUNT(DISTINCT obligoremploymentverificationcode) AS employment_verif,
       COUNT(DISTINCT obligorcreditscoretype) AS score_type, COUNT(DISTINCT vehiclenewusedcode) AS new_used
FROM {RAW} GROUP BY 1
''')

,shelf,originatorname,originator_value,subvented,underwritingindicator,employment_verif,score_type,new_used
0,Select,1,CBS,1,1,1,1,2
1,Prime,1,CBS,1,1,1,1,2


### A3. Credit profile at origination

In [11]:
fico_bins = [0, 550, 600, 650, 660, 720, 900]
fico_lbls = ["<550", "550-599", "600-649", "650-659", "660-719", "720+"]
first_obs["fico_band"] = pd.cut(first_obs.fico, fico_bins, labels=fico_lbls, right=False).astype(str).replace("nan", "missing")
first_obs["apr_band"] = pd.cut(first_obs.apr, [0, 6, 9, 12, 15, 18, 21, 40], right=False).astype(str)

def bal_mix(col):
    t = first_obs.pivot_table(index=col, columns="shelf", values="beg_bal", aggfunc="sum")
    return (t / t.sum()).style.format("{:.1%}").set_caption(f"Balance mix by {col}")

display(bal_mix("fico_band"))
display(bal_mix("apr_band"))

def wavg(g, c):
    m = g[c].notna()
    return np.average(g.loc[m, c], weights=g.loc[m, "beg_bal"])

first_obs.groupby("shelf").apply(lambda g: pd.Series({
    "loans": len(g), "balance_mm": g.beg_bal.sum() / 1e6,
    **{f"wa_{c}": wavg(g, c) for c in ["fico", "apr", "ltv", "pti", "term", "vehicle_age", "apr_spread"]},
    "pct_coobligor": g.coobligor.mean(), "pct_no_fico": g.fico.isna().mean(),
}), include_groups=False).T

shelf,Prime,Select
fico_band,,
550-599,4.6%,25.9%
600-649,10.6%,52.1%
650-659,4.3%,1.5%
660-719,25.5%,5.7%
720+,53.7%,0.8%
<550,0.8%,9.2%
missing,0.5%,4.9%


shelf,Prime,Select
apr_band,,
"[0, 6)",21.2%,0.0%
"[12, 15)",16.6%,24.7%
"[15, 18)",8.6%,43.3%
"[18, 21)",1.4%,18.6%
"[21, 40)",0.0%,4.0%
"[6, 9)",32.7%,1.7%
"[9, 12)",19.5%,7.7%


shelf,Prime,Select
loans,1.370881e+06,123917.000000
balance_mm,2.593868e+04,2447.628901
wa_fico,7.288571e+02,608.450478
wa_apr,9.360408e+00,15.997540
wa_ltv,9.519229e-01,1.043181
wa_pti,8.506670e+00,10.288202
wa_term,6.664838e+01,69.437130
wa_vehicle_age,4.057362e+00,4.523020
wa_apr_spread,9.328910e+00,15.952090
pct_coobligor,2.945726e-01,0.279163


### A4. Reference definition: CarMax's own tiers (10-K FY2026, 10-Q Q2 FY2027)

The tiers are defined by **who finances** the loan, not by credit score:

- **Tier 1 = CAF's own lending.** CAF reviews every application first, using proprietary underwriting.
- **Tier 2 / Tier 3 = third-party lenders.** CarMax collects a fee from Tier 2 lenders (or none is paid) and pays a fee to Tier 3 lenders.
- **CAF also lends into the Tier 2/3 credit band.** These are CAF's Tier 2/3 originations.

| Disclosure | Value | Source |
|---|---|---|
| CAF receivables, Tier 2/3 share | 3.4% at 2/28/26; 4.7% ($765.6M of $16.2B) at 8/31/26 | 10-K, 10-Q |
| CAF originations H1 FY27 | ~90% Tier 1 / ~10% Tier 2 | 10-Q |
| Scorecard grade mix at 2/28/26 | Tier 1: A 58.2%, B 31.1%, C & other 7.3%. Tier 2/3: C & other 3.4% | 10-K |
| FICO in the scorecard | "not a significant factor" | 10-K, 10-Q |
| Allowance rate at 2/28/26 (calc.) | Tier 1 ≈ 2.4%; Tier 2/3 ≈ 13.7% ($75.3M / $550.4M) | 10-K |
| Non-prime securitization, 24 Sep 2025 | ~$930M sold off balance sheet. Allowance release **$30.3M Tier 1 / $11.9M Tier 2/3** | 10-K |
| Non-prime securitization, Q2 FY27 | ~$596M sold; held-for-sale transfers in Q2 were entirely Tier 2/3 ($12.5M) | 10-Q |
| FY27 plan | Raise CAF's share of Tier 2 volume from <15% to ~30% | 10-K |

### A5. Reference definition: prospectus pool criteria, reconciled to our data

The prospectus wording, verbatim:

- **2025-A and 2025-B:** *"selected from (i) CarMax Business Services' core portfolio … with a FICO® score at origination less than 650, except for a small portion of receivables from the core portfolio with a FICO® score at origination greater than or equal to 650 that have either alternate internal credit grades or original terms greater than 72 months, and (ii) receivables originated outside of CarMax Business Services' core portfolio."*
- **2024-A:** the same, but **without** the FICO ≥ 650 exception.

In [12]:
prospectus = pd.DataFrame([
    # trust, cutoff, pool balance, receivables, WA FICO, WA APR, WA orig term  (424B5)
    ("CarMax Select Receivables Trust 2024-A", "2024-05-31", 666_668_280.61, 32_816, 603.1, 16.06, 68.45),
    ("CarMax Select Receivables Trust 2025-A", "2025-02-28", 826_876_936.50, 42_299, 608.4, 15.96, 68.93),
    ("CarMax Select Receivables Trust 2025-B", "2025-08-31", 923_081_739.63, 48_800, 612.4, 16.03, 69.41),
], columns=["trust_name", "cutoff", "p_pool_bal", "p_loans", "p_wa_fico", "p_wa_apr", "p_wa_term"]).set_index("trust_name")

ours = sel.groupby("trust_name").apply(lambda g: pd.Series({
    "cutoff": str(g.cutoff_period.min()), "end_bal": g.end_bal.sum(), "loans": len(g),
    "wa_fico": wavg(g, "fico"), "wa_apr": wavg(g, "apr"), "wa_term": wavg(g, "term"),
}), include_groups=False)

rec = prospectus.join(ours, rsuffix="_ours")
rec["bal_diff_pct"] = rec.end_bal / rec.p_pool_bal - 1
rec["loan_diff"] = rec.loans - rec.p_loans
rec

,cutoff,p_pool_bal,p_loans,p_wa_fico,p_wa_apr,p_wa_term,cutoff_ours,end_bal,loans,wa_fico,wa_apr,wa_term,bal_diff_pct,loan_diff
trust_name,,,,,,,,,,,,,,
CarMax Select Receivables Trust 2024-A,2024-05-31,6.666683e+08,32816,603.1,16.06,68.45,2024-05-31,6.666683e+08,32818,603.103714,16.045052,68.901080,0.0,2
CarMax Select Receivables Trust 2025-A,2025-02-28,8.268769e+08,42299,608.4,15.96,68.93,2025-02-28,8.268769e+08,42299,608.408985,15.948259,69.376592,0.0,0
CarMax Select Receivables Trust 2025-B,2025-08-31,9.230817e+08,48800,612.4,16.03,69.41,2025-08-31,9.230817e+08,48800,612.442574,16.007385,69.878166,0.0,0


### A6. Tier 1 test

**(a) Observable proxy: does each loan fit the core-portfolio rule?** The buckets are:
- FICO < 650 → consistent with core (Tier 1).
- FICO ≥ 650 and term > 72 → a core exception (2025 deals only).
- FICO ≥ 650 and term ≤ 72 → either an "alternate internal grade" or non-core. For 2024-A, which has no exception, these **must** be non-core.
- Missing FICO → unknown; typically a thin file.

In [14]:
def core_bucket(r):
    if pd.isna(r.fico):  return "no FICO"
    if r.fico < 650:     return "FICO<650 (core rule)"
    if r.term > 72:      return "FICO>=650, term>72 (core exception)"
    return "FICO>=650, term<=72 (alt grade or non-core)"

sel = first_obs[first_obs.shelf == "Select"].copy()
sel["core_bucket"] = sel.apply(core_bucket, axis=1)
mix = sel.pivot_table(index="core_bucket", columns="trust_name", values="beg_bal", aggfunc="sum")
display((mix / mix.sum()).style.format("{:.1%}").set_caption("Select pool balance by core-rule bucket"))
sel.groupby("core_bucket").apply(lambda g: pd.Series({"wa_apr": wavg(g, "apr"), "wa_ltv": wavg(g, "ltv"), "wa_pti": wavg(g, "pti")}), include_groups=False)

trust_name,CarMax Select Receivables Trust 2024-A,CarMax Select Receivables Trust 2025-A,CarMax Select Receivables Trust 2025-B
core_bucket,,,
FICO<650 (core rule),91.6%,87.3%,83.9%
"FICO>=650, term<=72 (alt grade or non-core)",3.8%,3.0%,1.7%
"FICO>=650, term>72 (core exception)",1.3%,4.6%,8.7%
no FICO,3.4%,5.1%,5.7%


,wa_apr,wa_ltv,wa_pti
core_bucket,,,
FICO<650 (core rule),16.003089,1.045580,9.970918
"FICO>=650, term<=72 (alt grade or non-core)",15.345940,1.081087,12.428192
"FICO>=650, term>72 (core exception)",13.766274,1.058128,10.249328
no FICO,18.671404,0.962980,14.822312


**(b) CarMax's own definition, implied by the 10-K allowance release on the Sept 2025 sale (Select 2025-B).**

- Tier 2/3 balance ≈ Tier 2/3 allowance released ÷ Tier 2/3 allowance rate.
- The rate held at the time of sale is not disclosed, so the result is shown across a range of rates.

In [15]:
pool_bal = 930e6          # "approximately $930 million" sold (10-K)
t23_release = 11.9e6      # allowance released, Tier 2/3 (10-K)
rates = [0.10, 0.12, 0.137, 0.15, 0.20]
pd.DataFrame({"t23_allowance_rate": rates,
              "implied_t23_bal_mm": [t23_release / r / 1e6 for r in rates],
              "implied_t23_share": [t23_release / r / pool_bal for r in rates]}
            ).assign(implied_tier1_share=lambda d: 1 - d.implied_t23_share).style.format(
    {"t23_allowance_rate": "{:.1%}", "implied_t23_bal_mm": "{:.0f}", "implied_t23_share": "{:.1%}", "implied_tier1_share": "{:.1%}"})

,t23_allowance_rate,implied_t23_bal_mm,implied_t23_share,implied_tier1_share
0,10.0%,119,12.8%,87.2%
1,12.0%,99,10.7%,89.3%
2,13.7%,87,9.3%,90.7%
3,15.0%,79,8.5%,91.5%
4,20.0%,60,6.4%,93.6%


**Verdict (A).**

**CarMax's own definition supports the view that Select is largely Tier 1.**
- The Select pools are mostly core-portfolio (CAF Tier 1) loans.
- The 10-K allowance split implies roughly **90% Tier 1 / 10% Tier 2/3** for 2025-B.
- The loan-level core-rule test places **84–92%** of balance in the FICO < 650 core bucket, consistent with that split.

**A pure FICO definition does not support it.**
- If "Tier 1" meant prime credit (e.g. FICO ≥ 660), the view fails: only about 5% of Select loans clear that bar.
- These are Tier 1 by *originator*, not by credit quality.

**Gaps**
- The core vs non-core split is **not disclosed** per trust in the text I could extract. Check the full prospectus.
- Tier cannot be assigned loan by loan.
- For 2024-A, FICO ≥ 650 loans give a hard *lower bound* on the non-core share.
- The Q2 FY27 deal (~$596M, likely 2026-A/B) came after CAF's Tier 2 expansion, and its Q2 held-for-sale transfers were all Tier 2/3. Newer pools may therefore be far less Tier 1. Our data does not cover 2026 trusts.

## Part B — Ingested CarMax datasets, and linking loans to trusts
### B1. Inventory

In [ ]:
inv = try_q('''
SELECT table_catalog AS db, table_schema AS sch, table_name AS name, table_type, row_count,
       created::DATE AS created, last_altered::DATE AS last_altered, table_owner
FROM SNOWFLAKE.ACCOUNT_USAGE.TABLES
WHERE deleted IS NULL
  AND (table_name ILIKE '%CARMAX%' OR table_name ILIKE '%KMX%' OR table_schema ILIKE '%KMX%' OR table_schema ILIKE '%CARMAX%')
ORDER BY 1, 2, 3
''')
inv

In [ ]:
# Upstream sources used by the monorepo's carmax job (workflows/jobs/carmax) - access depends on role
display(try_q("SELECT 'srs_centralized_kpis' AS src, COUNT(*) AS n, MIN(period_end_date) AS mn, MAX(period_end_date) AS mx FROM srs_mapped.common.srs_centralized_kpis WHERE srs_ticker = 'KMX'"))
display(try_q("SELECT 'mscience data_joined' AS src, COUNT(*) AS n, MIN(start_date) AS mn, MAX(end_date) AS mx FROM m_science_unified_data_model.unified_data_model.data_joined WHERE ticker_symbol = 'KMX'"))
display(try_q("SELECT 'yipit kmx_core_metrics_view' AS src, COUNT(*) AS n, MIN(period_end) AS mn, MAX(period_end) AS mx FROM srs_yipit_from_share.yipitdata.kmx_core_metrics_view"))

| Dataset | Grain | Trust key | Use |
|---|---|---|---|
| `BOTS.AUTOABS.CARMAX_DETAIL` | loan × month, from each trust's inception | `TRUST_NAME` | Main history (A, C) |
| `BOTS.AUTOABS.CARMAX_SEC2_DETAIL` | loan × month, 2025-08 → 2026-01 | `TRUST_NAME` | Latest months |
| `BOTS.AUTOABS.CARMAX_SEC2_LIST`, `BOTS.KMX.LIST` | filing | `TRUST_NAME`, CIK in `SEC_URL` | Filing index / EDGAR links |
| `BOTS.KMX.DETAIL` → `ABS_COMBINED[_PROCESSED]` | loan × month (2025-08 → 2025-12 for CarMax) | `TRUST_NAME` | Multi-issuer comparison, derived fields |
| `BOTS.KMX.AUTOABS_*_DETAIL` | loan × month, other issuers | `TRUST_NAME` | Peers (Ally, Carvana, Exeter, …) |
| `BOTS.KMX.FED_FUND_RATES` | daily | — | APR spread |
| `MSCIENCE.KMX.KPIS` / `SIGNALS` | company KPI / daily signal | **none** | Retail units, sales, inventory |
| M Science `data_joined`, YipitData `kmx_core_metrics_view`, `srs_centralized_kpis` | aggregate | **none** | Upstream of `MSCIENCE.KMX` |

The retail and alt-data sets cannot be tied to loans or trusts; they can only be lined up by month.

### B2. Can each loan be tied to its trust?

In [16]:
checks = []
for name, t, where in [("RAW", RAW, "1=1"), ("SEC2", SEC2, "1=1"), ("KMX.DETAIL", KMXD, "1=1"), ("PROC (CarMax)", PROC, "trust_name ILIKE '%carmax%'")]:
    checks.append(q(f'''
    SELECT '{name}' AS src, COUNT(*) AS n_rows, COUNT(trust_name) / COUNT(*) AS trust_fill,
           COUNT(DISTINCT trust_name) AS trusts,
           COUNT(DISTINCT trust_name, assetnumber, reportingperiodendingdate) AS uniq_keys,
           1 - uniq_keys / n_rows AS dup_rate
    FROM {t} WHERE {where}'''))
pd.concat(checks)

,src,n_rows,trust_fill,trusts,uniq_keys,dup_rate
0,RAW,38256591,1.0,20,37272024,0.025736
0,SEC2,8286235,1.0,20,8085967,0.024169
0,KMX.DETAIL,6865501,1.0,20,6731983,0.019448
0,PROC (CarMax),6865501,1.000000,20,6731983,0.019448


In [ ]:
# Asset numbers reused across trusts -> key must be (trust_name, assetnumber)
collide = q(f'''
WITH a AS (SELECT DISTINCT trust_name, assetnumber FROM {RAW}),
     m AS (SELECT assetnumber FROM a GROUP BY 1 HAVING COUNT(DISTINCT trust_name) > 1)
SELECT a.trust_name, COUNT(*) AS colliding_assetnumbers
FROM a JOIN m USING (assetnumber) GROUP BY 1 ORDER BY 2 DESC
''')
print(f"asset numbers appearing in >1 trust: {first_obs.assetnumber.duplicated(keep=False).sum():,} loan records")
collide

In [ ]:
# Are colliding asset numbers the same loan (same origination) or reused IDs?
q(f'''
WITH f AS (
    SELECT trust_name, assetnumber, originationdate, originalloanamount FROM {RAW}
    QUALIFY ROW_NUMBER() OVER (PARTITION BY trust_name, assetnumber ORDER BY reportingperiodendingdate) = 1
), m AS (SELECT assetnumber FROM f GROUP BY 1 HAVING COUNT(*) > 1)
SELECT COUNT(DISTINCT assetnumber) AS colliding,
       COUNT(DISTINCT IFF(n_orig = 1 AND n_amt = 1, assetnumber, NULL)) AS same_loan_attributes
FROM (SELECT assetnumber, COUNT(DISTINCT originationdate) AS n_orig, COUNT(DISTINCT originalloanamount) AS n_amt
      FROM f JOIN m USING (assetnumber) GROUP BY 1)
''')

In [ ]:
# Overlap of the four loan-level tables by trust x period (row counts)
ov = q(f'''
SELECT src, trust_name, reportingperiodendingdate AS period, COUNT(*) AS n FROM (
    SELECT 'RAW' AS src, trust_name, reportingperiodendingdate FROM {RAW}
    UNION ALL SELECT 'SEC2', trust_name, reportingperiodendingdate FROM {SEC2}
    UNION ALL SELECT 'KMX.DETAIL', trust_name, reportingperiodendingdate FROM {KMXD}
    UNION ALL SELECT 'PROC', trust_name, reportingperiodendingdate FROM {PROC} WHERE trust_name ILIKE '%carmax%'
) WHERE reportingperiodendingdate >= '2025-08-01'
GROUP BY 1, 2, 3
''')
ov.pivot_table(index=["trust_name", "period"], columns="src", values="n").loc[lambda d: d.index.get_level_values(0).str.contains("Select")]

### B3. Coverage vs SEC EDGAR (checked 2026-10-06)

- **Last filing loaded:** 2026-02-17, loaded 2026-02-20 (`CARMAX_SEC2_LIST`).
- **Months still on EDGAR:** EDGAR shows monthly ABS-EE filings continuing through **2026-09-15**. The March–September 2026 filings (loan data for February–August 2026) are missing.
- **Trusts not ingested:**
  - Prime: Auto Owner Trust 2026-1, 2026-2, 2026-3, 2026-4.
  - Non-prime: Select Receivables Trust 2026-A, 2026-B, 2026-C.

In [17]:
q(f"SELECT trust_name, MAX(TO_DATE(filed_date, 'MM/DD/YY')) AS last_filed_loaded FROM BOTS.AUTOABS.CARMAX_SEC2_LIST GROUP BY 1 ORDER BY 2, 1")

,trust_name,last_filed_loaded
0,CarMax Auto Owner Trust 2021-4,2025-11-17
1,CarMax Auto Owner Trust 2022-1,2026-02-17
2,CarMax Auto Owner Trust 2022-2,2026-02-17
3,CarMax Auto Owner Trust 2022-3,2026-02-17
4,CarMax Auto Owner Trust 2022-4,2026-02-17
5,CarMax Auto Owner Trust 2023-1,2026-02-17
6,CarMax Auto Owner Trust 2023-2,2026-02-17
7,CarMax Auto Owner Trust 2023-3,2026-02-17
8,CarMax Auto Owner Trust 2023-4,2026-02-17
9,CarMax Auto Owner Trust 2024-1,2026-02-17


## Part C — Tier visibility and modelling
### C1. Is there a tier or grade field anywhere?

In [18]:
try_q('''
SELECT table_catalog, table_schema, table_name, column_name FROM SNOWFLAKE.ACCOUNT_USAGE.COLUMNS
WHERE deleted IS NULL
  AND (table_name ILIKE '%CARMAX%' OR table_schema ILIKE '%KMX%' OR table_name ILIKE '%KMX%')
  AND (column_name ILIKE '%TIER%' OR column_name ILIKE '%GRADE%' OR column_name ILIKE '%SCORE%' OR column_name ILIKE '%PRIME%')
ORDER BY 1, 2, 3, 4
''')

,table_catalog,table_schema,table_name,column_name
0,BOTS,AUTOABS,CARMAX_DETAIL,OBLIGORCREDITSCORE
1,BOTS,AUTOABS,CARMAX_DETAIL,OBLIGORCREDITSCORETYPE
2,BOTS,AUTOABS,CARMAX_SEC2_DETAIL,OBLIGORCREDITSCORE
3,BOTS,AUTOABS,CARMAX_SEC2_DETAIL,OBLIGORCREDITSCORETYPE
4,BOTS,KMX,ABS_COMBINED,OBLIGORCREDITSCORE
5,BOTS,KMX,ABS_COMBINED,OBLIGORCREDITSCORETYPE
6,BOTS,KMX,ABS_COMBINED_PROCESSED,OBLIGORCREDITSCORE
7,BOTS,KMX,ABS_COMBINED_PROCESSED,OBLIGORCREDITSCORETYPE
8,BOTS,KMX,AUTOABS_ALLY_DETAIL,OBLIGORCREDITSCORE
9,BOTS,KMX,AUTOABS_ALLY_DETAIL,OBLIGORCREDITSCORETYPE


Only `OBLIGORCREDITSCORE` and `OBLIGORCREDITSCORETYPE` exist. **There is no loan-level tier or grade**, because the SEC loan-level format (EX-102) does not require one. CAF's A/B/C grades are only disclosed in aggregate in the 10-K and 10-Q.

### C2. Proxy model: Select shelf vs prime shelf

- **Target:** the loan sits on the Select shelf. This is the best label we can observe. It stands for "CAF lower-credit book (core FICO < 650 + Tier 2/3)", **not** Tier 2/3 itself.
- **Data:** trusts with cutoffs from 2024 onwards, for vintage comparability.
- **Validation:** Select 2025-B and Auto Owner 2025-4 are held out as the test set.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import roc_auc_score

feats = ["fico", "apr", "apr_spread", "ltv", "pti", "term", "coobligor", "vehicle_age", "income_verif"]
d = first_obs[pd.to_datetime(first_obs.cutoff_period) >= "2024-01-01"].copy()
d["no_fico"] = d.fico.isna().astype(int)
X_cols = feats + ["no_fico"]
d["y"] = (d.shelf == "Select").astype(int)
test_trusts = ["CarMax Select Receivables Trust 2025-B", "CarMax Auto Owner Trust 2025-4"]
tr, te = d[~d.trust_name.isin(test_trusts)], d[d.trust_name.isin(test_trusts)]
print(d.groupby("shelf").size().to_dict(), "| test trusts present:", te.trust_name.unique().tolist())

logit = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
logit.fit(tr[X_cols], tr.y)
print("logit AUC  train %.3f | held-out %.3f" % (roc_auc_score(tr.y, logit.predict_proba(tr[X_cols])[:, 1]),
                                                roc_auc_score(te.y, logit.predict_proba(te[X_cols])[:, 1])))
pd.Series(logit[-1].coef_[0], index=X_cols).sort_values(key=abs, ascending=False).rename("std_coef").to_frame()

In [ ]:
# Shallow tree for readable cut-offs
imp = SimpleImputer(strategy="constant", fill_value=-1)
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=500, class_weight="balanced")
tree.fit(imp.fit_transform(tr[X_cols]), tr.y)
print("tree AUC held-out %.3f" % roc_auc_score(te.y, tree.predict_proba(imp.transform(te[X_cols]))[:, 1]))
print(export_text(tree, feature_names=X_cols, decimals=1))

### C3. Calibration: what the model implies for the Select pools

- Score each Select loan with the logistic model.
- Loans scoring as "prime-like" (p < 0.5) resemble core prime-shelf loans. These are most likely Tier 1 C-grade / alternate-grade loans.
- Compare the share of balance by score band with the ~90% Tier 1 implied by the 10-K (A6b).

In [ ]:
s = d[d.shelf == "Select"].copy()
s["p_select"] = logit.predict_proba(s[X_cols])[:, 1]
s["band"] = pd.cut(s.p_select, [0, .25, .5, .75, .9, 1], include_lowest=True).astype(str)
t = s.pivot_table(index="band", columns="trust_name", values="beg_bal", aggfunc="sum")
display((t / t.sum()).style.format("{:.1%}").set_caption("Select balance by modelled P(Select)"))

fig, ax = plt.subplots(figsize=(9, 3.5))
for shelf, g in d.assign(p=logit.predict_proba(d[X_cols])[:, 1]).groupby("shelf"):
    ax.hist(g.p, bins=50, alpha=.5, density=True, label=shelf)
ax.set(title="Modelled P(Select shelf) by actual shelf", xlabel="P(Select)"); ax.legend();

### C4. Limits

- **Shelf ≠ tier.** The Select shelf mixes Tier 1 core (FICO < 650) loans with Tier 2/3 loans. There is no loan-level label to train a Tier 1 vs Tier 2/3 model.
- **CAF's scorecard is unobservable,** and FICO is "not a significant factor" in it. So any FICO-led proxy will only approximate CAF's grades.
- **What we can do:** constrain Tier 2/3 to an aggregate per pool, using the 10-K allowance split and the 2024-A FICO ≥ 650 lower bound.
- **Rate scaling is confirmed.** `ORIGINALINTERESTRATEPERCENTAGE` and `PAYMENTTOINCOMEPERCENTAGE` are stored as fractions (0.16 = 16%). Our WA APR matches the prospectus contract rates.

## Data gaps

| # | Gap | Impact | Remedy |
|---|---|---|---|
| 1 | No loan-level tier or grade in EX-102 or our tables | Tier 1 vs 2/3 is only identifiable in aggregate | 10-K/10-Q allowance and grade tables; prospectus criteria; proxy model (C) |
| 2 | Prospectus doesn't quantify the core vs non-core split (text checked so far) | Can't pin the Tier 2/3 share per trust | Read the full 424B5 (the "pool composition" and "static pool" annexes); ask CarMax IR or the deal's rating-agency presale reports |
| 3 | Feed stopped Feb 2026 (last filing loaded 2026-02-17) | Missing loan data Feb–Aug 2026 and all 2026 trusts, including post-Tier-2-expansion Select deals | `BOTS` owner restarts the loader and adds the new trusts; or run a one-off EDGAR pull |
| 4 | Loader has no owner in the srs monorepo | Nobody is maintaining it | Identify the owner (Snowflake `table_owner` in B1) |
| 5 | `ORIGINATORNAME` is constant ("CBS") | Can't separate core from non-core origination | None in EX-102 |
| 6 | 3–8% of Select loans have no FICO | Bias in the FICO-based tests; likely skewed to non-core | Treat as a separate bucket; model with a `no_fico` flag |
| 7 | ~2.6% duplicate rows on (trust, asset, period) in raw tables | Overstated balances if not removed | `QUALIFY ROW_NUMBER()` on (trust, asset, period) |
| 8 | Asset numbers reused across trusts | Wrong joins on `assetnumber` alone | Always key on (trust_name, assetnumber) |